# LOGOS — GPU Training v2 (50MB TinyStories)
**Vedic GEMM + Langevin Dynamics | T4 16GB VRAM Safe**

| Config | Value | VRAM Cost |
|--------|-------|----------|
| d_model | 128 | — |
| Layers | 4 | — |
| Heads | 8 | — |
| Seq len | 128 | — |
| Vocab | ~4096 | — |
| **Model size** | **~15MB** | **~60MB VRAM** |
| Batch size | 1 | safe |
| Dataset | 50MB TinyStories | — |
| Epochs | 3 | — |
| **Expected** | **Loss 3.x** | **~4-6 hr** |

In [ ]:
# CELL 1 — GPU + VRAM Check
import os, subprocess, re, math, time, glob, json
import matplotlib.pyplot as plt

WORK_DIR   = '/kaggle/working/LOGOS'
TRAIN_FILE = '/kaggle/working/dataset_50mb.txt'
LOG_FILE   = '/kaggle/working/train_log.txt'

print('=== GPU Info ===')
os.system('nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader')

print('\n=== VRAM Budget ===')
r = subprocess.run(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'],
                   capture_output=True, text=True)
vram_mb = int(r.stdout.strip().split('\n')[0].strip())
print(f'Total VRAM  : {vram_mb} MB')
print(f'Model       : ~60 MB   (d=128 L=4)')
print(f'Gradients   : ~60 MB   (same as model)')
print(f'Activations : ~50 MB   (seq=128 batch=1)')
print(f'Velocity    : ~60 MB   (Langevin momentum)')
print(f'Overhead    : ~200 MB  (CUDA runtime)')
total_est = 60+60+50+60+200
print(f'Total Est   : ~{total_est} MB')
print(f'Free buffer : ~{vram_mb - total_est} MB')
if vram_mb - total_est > 1000:
    print('✅ VRAM sufficient — safe to train')
else:
    print('⚠️  Low VRAM — reduce d_model to 64')

print('\n=== CUDA ===')
os.system('nvcc --version | grep release')
print('\n=== RAM ===')
os.system('free -h | head -2')

In [ ]:
# CELL 2 — Dataset: 50MB TinyStories
TARGET_MB    = 50
TARGET_BYTES = TARGET_MB * 1024 * 1024

if os.path.exists(TRAIN_FILE) and os.path.getsize(TRAIN_FILE) >= TARGET_BYTES // 2:
    sz = os.path.getsize(TRAIN_FILE) // 1024 // 1024
    print(f'✅ dataset_50mb.txt exists: {sz} MB')
    with open(TRAIN_FILE) as f: print(f.read(300))
else:
    done = False

    # 1. Local .txt
    for fp in glob.glob('/kaggle/input/**/*.txt', recursive=True):
        if os.path.getsize(fp) > TARGET_BYTES // 2:
            print(f'Using local: {fp}')
            with open(fp,'r',errors='ignore') as fin, open(TRAIN_FILE,'w') as fout:
                fout.write(fin.read(TARGET_BYTES))
            done = True; break

    # 2. Local .jsonl
    if not done:
        for fp in glob.glob('/kaggle/input/**/*.jsonl', recursive=True):
            if os.path.getsize(fp) > TARGET_BYTES // 2:
                print(f'Using JSONL: {fp}')
                written = 0
                with open(TRAIN_FILE,'w') as fout:
                    with open(fp,'r',errors='ignore') as fin:
                        for line in fin:
                            if written >= TARGET_BYTES: break
                            try:
                                obj = json.loads(line)
                                t = obj.get('story', obj.get('text',''))
                                if t:
                                    fout.write(t.strip()+'\n\n')
                                    written += len(t)
                            except: pass
                done = True; break

    # 3. Download 50MB (wget — needs internet on Kaggle)
    if not done:
        print('Downloading 50MB from HuggingFace...')
        tmp = '/kaggle/working/_tmp_full.txt'
        ret = subprocess.run(
            ['wget','-q','--timeout=120',
             'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt',
             '-O', tmp],
            capture_output=True)
        if ret.returncode == 0 and os.path.exists(tmp):
            with open(tmp,'r',errors='ignore') as fin, open(TRAIN_FILE,'w') as fout:
                fout.write(fin.read(TARGET_BYTES))
            os.remove(tmp)
            done = True
            print('Downloaded ok')
        else:
            print(f'Download failed: {ret.stderr.decode()[:100]}')

    # 4. Reuse 5MB file ×10
    if not done:
        small = '/kaggle/working/dataset.txt'
        if os.path.exists(small):
            print('Reusing 5MB file x10...')
            with open(small) as fin:
                text = fin.read()
            with open(TRAIN_FILE,'w') as fout:
                written = 0
                while written < TARGET_BYTES:
                    fout.write(text)
                    written += len(text)
            done = True

sz = os.path.getsize(TRAIN_FILE)
print(f'\n✅ Dataset: {sz//1024//1024} MB ({sz:,} bytes)')

In [ ]:
# CELL 3 — Clone / Pull LOGOS
REPO_URL = 'https://github.com/Vikas8719/LOGOS.git'
if not os.path.exists(WORK_DIR):
    print('Cloning...')
    os.system(f'git clone {REPO_URL} {WORK_DIR}')
else:
    print('Pulling...')
    os.system(f'git -C {WORK_DIR} fetch origin main')
    os.system(f'git -C {WORK_DIR} reset --hard origin/main')

os.chdir(WORK_DIR)
os.system('git log --oneline -3')
os.system('ls src/ cuda/')

In [ ]:
# CELL 4 — Patch main.cpp: d=128 L=4 H=8 seq=128 epochs=3
# VRAM-safe config for T4 16GB
MAIN_CPP = f'{WORK_DIR}/src/main.cpp'
with open(MAIN_CPP) as f:
    src = f.read()

src = re.sub(r'cfg\.d_model\s*=\s*\d+;',    'cfg.d_model     = 128;', src)
src = re.sub(r'cfg\.num_layers\s*=\s*\d+;',  'cfg.num_layers  = 4;',   src)
src = re.sub(r'cfg\.num_heads\s*=\s*\d+;',   'cfg.num_heads   = 8;',   src)
src = re.sub(r'cfg\.max_seq_len\s*=\s*\d+;', 'cfg.max_seq_len = 128;', src)
src = re.sub(r'int SEQ\s*=\s*\d+;',          'int SEQ   = 128;',       src)
src = re.sub(r'int EPOCHS\s*=\s*\d+;',       'int EPOCHS = 3;',        src)
src = re.sub(r'LangevinOptimizer langevin\([\de.+\-]+f',
             'LangevinOptimizer langevin(2e-4f', src)

with open(MAIN_CPP, 'w') as f:
    f.write(src)

print('Patched config:')
for kw in ['d_model','num_layers','num_heads','int SEQ','int EPOCHS','LangevinOptimizer langevin']:
    for line in src.split('\n'):
        if kw in line and '=' in line and not line.strip().startswith('//'):
            print(' ', line.strip()); break

print('\nVRAM estimate:')
d, L, V, S = 128, 4, 4096, 128
model_mb = (V*d + S*d + d*V + L*(d*d*4 + d*4*d*2 + d*4)) * 4 / 1024 / 1024
print(f'  Model    : ~{model_mb:.0f} MB')
print(f'  Grads    : ~{model_mb:.0f} MB')
print(f'  Velocity : ~{model_mb:.0f} MB (Langevin)')
print(f'  Total    : ~{model_mb*3 + 250:.0f} MB  (fits T4 16GB ✅)')

In [ ]:
# CELL 5 — Build (CPU + GPU)
os.chdir(WORK_DIR)
os.system('rm -rf build && mkdir build')

r = subprocess.run(
    ['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],
    capture_output=True, text=True)
cap = r.stdout.strip().split('\n')[0].strip().replace('.', '')
print(f'GPU arch: sm_{cap}')

ret = os.system(f'''
cd {WORK_DIR} &&
cmake -B build -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_CXX_FLAGS="-O3 -march=native -std=c++20" \
  -DCMAKE_CUDA_ARCHITECTURES="{cap}" 2>&1 | tail -5 &&
cmake --build build --parallel $(nproc) 2>&1
''')

print(f'Build exit: {ret}')
os.system(f'ls -lh {WORK_DIR}/build/logos* 2>/dev/null')

GPU_BIN = f'{WORK_DIR}/build/logos_gpu'
CPU_BIN = f'{WORK_DIR}/build/logos'
BINARY  = GPU_BIN if os.path.exists(GPU_BIN) else CPU_BIN
MODE    = 'GPU' if os.path.exists(GPU_BIN) else 'CPU'
print(f'\nUsing: [{MODE}] {BINARY}')

In [ ]:
# CELL 6 — VRAM Monitor (background)
# Ye GPU memory track karta hai — crash se pehle warn karega
import threading, time

vram_log = []
stop_monitor = threading.Event()

def monitor_vram():
    while not stop_monitor.is_set():
        try:
            r = subprocess.run(
                ['nvidia-smi','--query-gpu=memory.used,memory.free',
                 '--format=csv,noheader,nounits'],
                capture_output=True, text=True)
            parts = r.stdout.strip().split(',')
            used  = int(parts[0].strip())
            free  = int(parts[1].strip())
            vram_log.append({'used': used, 'free': free, 't': time.time()})
            if free < 500:
                print(f'⚠️  VRAM LOW: {free}MB free — may OOM!')
        except: pass
        time.sleep(10)  # har 10 sec

monitor_thread = threading.Thread(target=monitor_vram, daemon=True)
monitor_thread.start()
print('✅ VRAM monitor started (every 10 sec)')
print('Will warn if < 500MB free')

In [ ]:
# CELL 7 — TRAIN (50MB, d=128 L=4, GPU)
# Auto-saves checkpoint har 1000 steps
# Kaggle session timeout se bachne ke liye frequent saves
os.chdir(WORK_DIR)

GPU_BIN = f'{WORK_DIR}/build/logos_gpu'
CPU_BIN = f'{WORK_DIR}/build/logos'
BINARY  = GPU_BIN if os.path.exists(GPU_BIN) else CPU_BIN
MODE    = 'GPU' if os.path.exists(GPU_BIN) else 'CPU'

ds_mb = os.path.getsize(TRAIN_FILE) // 1024 // 1024
print(f'╔══════════════════════════════════════════╗')
print(f'║  LOGOS v2 — Scale Training               ║')
print(f'║  Mode    : {MODE:<31}║')
print(f'║  Data    : {ds_mb} MB                           ║')
print(f'║  Model   : d=128 L=4 H=8 (~15M params)  ║')
print(f'║  Optim   : Langevin (NO Adam)            ║')
print(f'║  Target  : Loss < 5.0                    ║')
print(f'╚══════════════════════════════════════════╝')
print()

steps_log, losses_log, smooth_log = [], [], []
smooth = -1
last_checkpoint = 0
start_t = time.time()

proc = subprocess.Popen(
    [BINARY, '--train', TRAIN_FILE],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
    cwd=WORK_DIR
)

with open(LOG_FILE, 'w') as log:
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()

            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                s, l = int(m.group(1)), float(m.group(2))
                if not (math.isnan(l) or l > 20):
                    steps_log.append(s)
                    losses_log.append(l)
                    smooth = l if smooth < 0 else 0.95*smooth + 0.05*l
                    smooth_log.append(smooth)

            # VRAM status every 1000 steps
            if m and int(m.group(1)) % 1000 == 0 and vram_log:
                v = vram_log[-1]
                elapsed = (time.time()-start_t)/60
                print(f'  [VRAM: {v["used"]}MB used / {v["free"]}MB free | {elapsed:.0f}min]')

    except KeyboardInterrupt:
        proc.terminate()
        print('\n⏹️  Stopped — checkpoint saved')

stop_monitor.set()
proc.wait()
elapsed = time.time() - start_t

print(f'\nTime: {elapsed/60:.1f} min ({elapsed/3600:.1f} hr)')
if losses_log:
    drop = losses_log[0] - losses_log[-1]
    print(f'Start : {losses_log[0]:.4f}')
    print(f'Best  : {min(losses_log):.4f}')
    print(f'Final : {losses_log[-1]:.4f}')
    print(f'Drop  : {drop:.4f} nats')
    if drop > 2.0:   print('✅ Excellent learning!')
    elif drop > 1.0: print('✅ Good learning!')
    else:            print('⚠️  Need more steps')

In [ ]:
# CELL 8 — Loss Plot + VRAM Usage
if not steps_log and os.path.exists(LOG_FILE):
    smooth = -1
    with open(LOG_FILE) as f:
        for line in f:
            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                s, l = int(m.group(1)), float(m.group(2))
                if not (math.isnan(l) or l > 20):
                    steps_log.append(s); losses_log.append(l)
                    smooth = l if smooth<0 else 0.95*smooth+0.05*l
                    smooth_log.append(smooth)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('LOGOS v2 — 50MB TinyStories | d=128 L=4 | Langevin Optimizer', fontsize=12, fontweight='bold')

# Loss
axes[0].plot(steps_log, losses_log, 'steelblue', lw=0.4, alpha=0.3, label='Raw')
axes[0].plot(steps_log, smooth_log, 'crimson', lw=2, label='Smoothed')
if losses_log:
    axes[0].axhline(losses_log[0],  color='orange', ls='--', lw=1, label=f'Start={losses_log[0]:.2f}')
    axes[0].axhline(min(losses_log),color='green',  ls='--', lw=1, label=f'Best={min(losses_log):.2f}')
axes[0].set_title('Cross-Entropy Loss'); axes[0].set_xlabel('Steps')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Perplexity
perp = [math.exp(min(l, 10)) for l in smooth_log]
axes[1].plot(steps_log, perp, 'darkorchid', lw=1.5)
axes[1].fill_between(steps_log, perp, alpha=0.1, color='darkorchid')
axes[1].set_yscale('log')
if perp: axes[1].set_title(f'Perplexity: {perp[0]:.0f} → {perp[-1]:.0f}')
axes[1].set_xlabel('Steps'); axes[1].grid(alpha=0.3)

# VRAM
if vram_log:
    vt = [v['t'] - vram_log[0]['t'] for v in vram_log]
    vu = [v['used'] for v in vram_log]
    axes[2].plot(vt, vu, 'tomato', lw=1.5)
    axes[2].axhline(16000, color='red', ls='--', lw=1, label='T4 limit')
    axes[2].set_title('VRAM Usage (MB)'); axes[2].set_xlabel('Time (sec)')
    axes[2].legend(); axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'No VRAM data', ha='center', va='center',
                transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig('/kaggle/working/loss_curve_v2.png', dpi=150, bbox_inches='tight')
plt.show()

if losses_log:
    print(f'Drop: {losses_log[0]:.4f} → {losses_log[-1]:.4f} ({losses_log[0]-losses_log[-1]:.4f} nats)')

In [ ]:
# CELL 9 — Generate Text
os.chdir(WORK_DIR)
CPU_BIN = f'{WORK_DIR}/build/logos'

ckpts = sorted([f for f in glob.glob(f'{WORK_DIR}/*.bin') if 'vocab' not in f])
print(f'Checkpoints: {len(ckpts)} found')
if ckpts:
    latest = ckpts[-1]
    print(f'Using: {os.path.basename(latest)}\n')
    for p in [
        'Once upon a time',
        'The little girl named',
        'Tom had a red ball',
        'In a small village there',
    ]:
        print(f"Prompt: '{p}'")
        print('-'*50)
        os.system(f'{CPU_BIN} --generate {latest} "{p}" 2>&1')
        print()
else:
    print('No checkpoint — run Cell 7 first')

In [ ]:
# CELL 10 — Save All Outputs
import shutil
OUT_DIR = '/kaggle/working/logos_v2_output'
os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(WORK_DIR)

saved = []
# Best checkpoint only (largest step = latest)
ckpts = sorted([f for f in glob.glob(f'{WORK_DIR}/*.bin') if 'vocab' not in f])
if ckpts:
    shutil.copy(ckpts[-1], OUT_DIR); saved.append(os.path.basename(ckpts[-1]))
# Vocab
if os.path.exists(f'{WORK_DIR}/vocab.bin'):
    shutil.copy(f'{WORK_DIR}/vocab.bin', OUT_DIR); saved.append('vocab.bin')
# Plots + logs
for fp in ['/kaggle/working/train_log.txt',
           '/kaggle/working/loss_curve_v2.png']:
    if os.path.exists(fp):
        shutil.copy(fp, OUT_DIR); saved.append(os.path.basename(fp))

print(f'Saved {len(saved)} files to {OUT_DIR}:')
for s in saved: print(f'  {s}')
os.system(f'ls -lh {OUT_DIR}')
print('\n✅ Download from Output tab!')